# SKU-Node Correlation Analysis

Exploratory analysis of correlations between pricing, inventory, and sales at the SKU-Node level.
All statistical estimates include 95% confidence intervals and precision measures.

In [ ]:
# === PARAMETERS ===
END_DATE = "2026-03-25"
ROLLBACKS_PATH = None  # Set to rollbacks Excel path to exclude rollback SKUs
WAREHOUSE_ADDRESSES_PATH = r"C:\Users\valen\Desktop\WalmartPricing\Warehouse Addresses 03-25-2026 01-43-16 PM.csv"

# === IMPORTS ===
import sys, os, logging, warnings
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis.config import load_analysis_config
from src.analysis.data_prep import AnalysisDataPrep
from src.analysis import eda, statistical_tests, geographic_brand
from src.analysis import elasticity, did_effects, segmented
from src.analysis import optimization, simulation, strategy, summary

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

cfg = load_analysis_config()
ci_cfg = cfg["ci"]
print(f"Config loaded. CI level: {ci_cfg['level']}, Bootstrap samples: {ci_cfg['n_bootstrap']}")

In [ ]:
# === DATA PREPARATION ===
prep = AnalysisDataPrep(
    end_date=END_DATE,
    rollbacks_path=ROLLBACKS_PATH,
    warehouse_addresses_path=WAREHOUSE_ADDRESSES_PATH,
)
df = prep.run()
print(f"Master DataFrame: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

## Exploratory Data Analysis

Spearman correlation matrix with bootstrap CIs, scatter plots with regression CI bands, distributions with mean CIs.

In [ ]:
# Correlation matrix + top correlations with bootstrap CIs
corr_result = eda.compute_correlation_matrix(
    df,
    method=cfg["eda"]["correlation_method"],
    target_col=cfg["eda"]["target_col"],
    exclude_cols=cfg["eda"]["exclude_cols"],
    n_boot=ci_cfg["n_bootstrap"],
    ci_level=ci_cfg["level"],
    seed=ci_cfg["seed"],
)
eda.plot_correlation_heatmap(corr_result, top_n=cfg["eda"]["top_n_correlations"])

# Scatter plots for top features with CI bands and Spearman r + CI
top_features = corr_result["target_corrs"]["feature"].head(cfg["eda"]["top_n_scatter"]).tolist()
eda.plot_scatter_with_ci(
    df, top_features,
    target_col=cfg["eda"]["target_col"],
    n_boot=ci_cfg["n_bootstrap"],
    ci_level=ci_cfg["level"],
    seed=ci_cfg["seed"],
)

# Distributions with bootstrap CI of the mean
eda.plot_distributions(df, cfg["eda"]["distribution_cols"])

## Statistical Tests

Mann-Whitney U with effect sizes + bootstrap CIs, OLS with coefficient CI forest plots, price-revenue with bootstrap CIs.

In [ ]:
# Mann-Whitney: Price change impact
scfg = cfg["statistical_tests"]
price_test = statistical_tests.price_change_impact_test(
    df, **scfg["price_change"], n_boot=ci_cfg["n_bootstrap"],
    ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)

# Mann-Whitney: Inventory availability impact
inv_test = statistical_tests.inventory_impact_test(
    df, n_boot=ci_cfg["n_bootstrap"],
    ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)

# Margin decile analysis with bootstrap CI error bars
margin_deciles = statistical_tests.margin_decile_analysis(
    df, **scfg["margin_deciles"], n_boot=ci_cfg["n_bootstrap"],
    ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)

# OLS regression with coefficient CIs
feature_cols = [c for c in scfg["ols"]["feature_cols"] if c in df.columns]
ols_result = statistical_tests.ols_regression(
    df, feature_cols=feature_cols, ci_level=ci_cfg["level"],
)
ols_log = statistical_tests.ols_regression(
    df, feature_cols=feature_cols, log_transform=True, ci_level=ci_cfg["level"],
)

# Price change vs revenue with bootstrap CIs
pc_revenue = statistical_tests.price_change_revenue_analysis(
    df, n_boot=ci_cfg["n_bootstrap"], ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)

# Render all plots
statistical_tests.plot_statistical_tests(
    price_test, inv_test, margin_deciles, ols_result, ols_log, pc_revenue,
)

## Geographic & Brand Analysis

State/brand aggregations with bootstrap CI error bars, distribution breadth with Spearman r + CI.

In [ ]:
state_df = geographic_brand.state_sales_analysis(
    df, n_boot=ci_cfg["n_bootstrap"], ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)
brand_df = geographic_brand.brand_analysis(
    df, n_boot=ci_cfg["n_bootstrap"], ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)
breadth = geographic_brand.distribution_breadth_analysis(
    df, n_boot=ci_cfg["n_bootstrap"], ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)
geographic_brand.plot_geographic_brand(state_df, brand_df, breadth)

## Extended Feature Engineering

Add tire size (from `warehouse.d_tireproduct`) and MAP tire flag.

In [ ]:
df = prep.load_extended_features(df)
print(f"Columns after extended features: {df.shape[1]}")

## Price Elasticity Analysis

Generic log-log OLS elasticity by state, brand-state, city, and brand -- all with analytical CIs from OLS standard errors.

In [ ]:
ecfg = cfg["elasticity"]
df_elast = df[(df["qty_sold"] > 0) & (df["offer_price"] > 0)].copy()

# State-level elasticity with CIs
state_e = elasticity.estimate_elasticity(
    df_elast, ["State"], min_obs=ecfg["min_obs_state"], ci_level=ci_cfg["level"],
)
elasticity.plot_elasticity_bars(state_e, "State", title="State-Level Price Elasticity")

# Brand-State cross-tabulation with CIs
top_brands_e = df_elast.groupby("brand")["qty_sold"].sum().nlargest(ecfg["top_n_brands"]).index.tolist()
top_states_e = df_elast.groupby("State")["qty_sold"].sum().nlargest(ecfg["top_n_states"]).index.tolist()
df_bs_subset = df_elast[df_elast["brand"].isin(top_brands_e) & df_elast["State"].isin(top_states_e)]
bs_e = elasticity.estimate_elasticity(
    df_bs_subset, ["brand", "State"], min_obs=ecfg["min_obs_brand_state"], ci_level=ci_cfg["level"],
)
elasticity.plot_elasticity_heatmap(bs_e, "brand", "State", title="Brand-State Price Elasticity")

# City-level drill-down
city_e = elasticity.estimate_elasticity(
    df_bs_subset, ["brand", "State", "Town"], min_obs=ecfg["min_obs_city"], ci_level=ci_cfg["level"],
)
if len(city_e) > 0:
    city_var = city_e.groupby(["brand", "State"])["elasticity"].std().reset_index(name="std")
    notable = city_var[city_var["std"] > 0.5]
    if len(notable) > 0:
        print(f"Brand-State segments with high city-level variation (std > 0.5): {len(notable)}")
        for _, r in notable.iterrows():
            cities = city_e[(city_e["brand"] == r["brand"]) & (city_e["State"] == r["State"])]
            print(f"\n  {r['brand']} in {r['State']} (std={r['std']:.2f}):")
            for _, c in cities.sort_values("elasticity").iterrows():
                sig = "*" if c["p_value"] < 0.05 else ""
                print(f"    {c['Town']:25s} e={c['elasticity']:.3f} [{c['ci_lower']:.3f}, {c['ci_upper']:.3f}]{sig} (n={c['n_obs']:,})")
    else:
        print("No brand-state segments show meaningful city-level variation.")

# Brand sensitivity rankings with CIs
brand_rank = elasticity.brand_sensitivity_rankings(df_elast, min_obs=ecfg["min_obs_brand"], ci_level=ci_cfg["level"])
display(brand_rank.style.format({
    "elasticity": "{:.4f}", "se": "{:.4f}", "ci_lower": "{:.4f}", "ci_upper": "{:.4f}",
    "p_value": "{:.4f}", "avg_te_margin": "{:.1%}", "avg_wm_margin": "{:.1%}",
    "total_qty": "{:,.0f}", "total_revenue": "${:,.0f}",
}, na_rep="-").background_gradient(subset=["elasticity"], cmap="RdYlGn"))

## Seasonal Price Sensitivity

Sub-period elasticity with CI bands to detect time-varying price sensitivity.

In [ ]:
START_DATE = (pd.to_datetime(END_DATE) - pd.Timedelta(days=cfg["data_prep"]["analysis_days"] - 1)).strftime("%Y-%m-%d")

seasonal_e = elasticity.estimate_seasonal_elasticity(
    df, START_DATE, END_DATE,
    n_periods=ecfg.get("seasonal_n_periods", 3),
    brands=top_brands_e,
    min_obs_overall=ecfg["min_obs_state"],
    min_obs_brand=ecfg["min_obs_brand"],
    ci_level=ci_cfg["level"],
)
elasticity.plot_seasonal_elasticity(seasonal_e)

## Heterogeneous Treatment Effects (DiD)

Difference-in-differences with HC1 robust SEs and CIs, decomposed by brand, price tier, state, and MAP status.

In [ ]:
did_cfg = cfg["did"]
did_results = did_effects.run_all_did(
    df,
    dimensions=did_cfg["dimensions"],
    treatment_col=did_cfg["treatment_col"],
    max_controls_per_brand=did_cfg["max_controls_per_brand"],
    pre_window_days=did_cfg["pre_window_days"],
    post_window_days=did_cfg["post_window_days"],
    min_event_buffer_days=did_cfg["min_event_buffer_days"],
    top_n=did_cfg["top_n_per_dimension"],
    min_obs=did_cfg["min_obs"],
    ci_level=ci_cfg["level"],
)
did_effects.plot_did_results(did_results)

## Segmented Analysis

Tire size, MAP vs non-MAP, inventory visibility comparisons with effect sizes + CIs. Segmented elasticity with CI error bars.

In [ ]:
# Tire size analysis
tire_result = segmented.tire_size_analysis(
    df, n_boot=ci_cfg["n_bootstrap"], ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)

# MAP vs Non-MAP comparison
map_df = segmented.map_vs_nonmap_comparison(
    df, n_boot=ci_cfg["n_bootstrap"], ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)

# Inventory visibility
vis_df = segmented.inventory_visibility_analysis(
    df, n_boot=ci_cfg["n_bootstrap"], ci_level=ci_cfg["level"], seed=ci_cfg["seed"],
)

# Segmented elasticity
seg_elast = {}
for seg_item in [{"col": "is_MAP_tire"}, {"col": "can_show_inventory"}, {"col": "tire_diameter"}]:
    col = seg_item["col"]
    if col in df.columns:
        seg_elast[col] = segmented.segmented_elasticity(
            df, col, min_obs=cfg["segmented"]["min_obs"], ci_level=ci_cfg["level"],
        )

segmented.plot_segmented_results(tire_result, map_df, vis_df, seg_elast)

## Optimal Margin Targeting

Quadratic margin-sales/profit fits with delta-method CIs for the optimal margin vertex.

In [ ]:
ocfg = cfg["optimization"]
top_brands_opt = brand_rank["brand"].head(ocfg["top_n_brands"]).tolist() if len(brand_rank) > 0 else top_brands_e

margin_opt = optimization.margin_sales_optimization(
    df, top_brands_opt,
    margin_cols=ocfg["margin_cols"],
    margin_bounds=tuple(ocfg["margin_bounds"]),
    optimal_range=tuple(ocfg["optimal_range"]),
    min_obs=ocfg["min_obs"],
    ci_level=ci_cfg["level"],
)
profit_opt = optimization.profit_maximizing_margin(
    df, top_brands_opt,
    margin_bounds=tuple(ocfg["margin_bounds"]),
    optimal_range=tuple(ocfg["optimal_range"]),
    min_obs=ocfg["min_obs"],
    ci_level=ci_cfg["level"],
)
optimization.plot_optimization_results(margin_opt, profit_opt)

## What-If Price Change Simulation

Simulated price changes with CI bands propagated from elasticity SE uncertainty.

In [ ]:
sim_cfg = cfg["simulation"]
pct_changes = np.arange(sim_cfg["pct_changes_start"], sim_cfg["pct_changes_end"], sim_cfg["pct_changes_step"])

df_sim = simulation.simulate_price_changes(
    brand_rank, df, pct_changes=pct_changes, min_obs_brand=sim_cfg["min_obs_brand"],
)
simulation.plot_simulation_results(
    df_sim, brand_rank,
    top_n=sim_cfg["top_elastic_brands"],
)

sweet_spots = simulation.find_sweet_spots(df_sim, brand_rank)
if len(sweet_spots) > 0:
    print("\n=== Recommended Price Changes (Sweet Spot) ===")
    display(sweet_spots)

## Actionable Pricing Strategy

Master brand-state strategy table with CI-derived confidence levels and projected revenue impact ranges.

In [ ]:
stcfg = cfg["strategy"]
df_strategy = strategy.build_strategy_table(
    bs_e, brand_rank, margin_opt, did_results,
    elastic_threshold=stcfg["elastic_threshold"],
    inelastic_threshold=stcfg["inelastic_threshold"],
    max_recommended_change=stcfg["max_recommended_change"],
    high_confidence_min_n=stcfg["high_confidence_min_n"],
    medium_confidence_min_n=stcfg["medium_confidence_min_n"],
    confidence_threshold=stcfg["confidence_threshold"],
    ci_level=ci_cfg["level"],
)

if len(df_strategy) > 0:
    display(df_strategy.head(20).style.format({
        "elasticity": "{:.3f}", "elasticity_ci_lower": "{:.3f}", "elasticity_ci_upper": "{:.3f}",
        "treatment_effect": "{:+.4f}",
        "optimal_te_margin": "{:.1%}", "current_te_margin": "{:.1%}", "current_wm_margin": "{:.1%}",
        "recommended_change": "{:+.1%}",
        "expected_qty_impact": "{:+.1%}", "expected_revenue_impact": "{:+.1%}",
        "rev_impact_ci_lower": "{:+.1%}", "rev_impact_ci_upper": "{:+.1%}",
        "p_value": "{:.4f}",
    }, na_rep="-").background_gradient(subset=["expected_revenue_impact"], cmap="RdYlGn"))

strategy.generate_narrative(df_strategy)
strategy.plot_strategy_overview(df_strategy)

## Executive Summary

In [ ]:
all_results = {
    "state_elasticity": state_e,
    "brand_state_elasticity": bs_e,
    "brand_rank": brand_rank,
    "did_results": did_results,
    "seasonal": seasonal_e,
    "strategy": df_strategy,
    "corr_result": corr_result,
}
summary.generate_executive_summary(df, all_results, START_DATE, END_DATE)

## Save & Cleanup

In [ ]:
project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
output_dir = os.path.join(project_root, "outputs")
out_cfg = cfg["output"]

path = summary.save_analysis_dataset(
    df, output_dir, END_DATE, fmt=out_cfg["format"], fallback_fmt=out_cfg["fallback_format"],
)
print(f"Dataset saved to {path}")
print(f"Shape: {df.shape}")

prep.close()
print("Done.")